In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

file_path = (
    PROJECT_DIR
    / "data"
    / "Final_Version"
    / "Raw"
    / "Loans.csv"
)
print("File exists:", file_path.exists())

df = pd.read_csv(file_path)

print("Rows saved:", len(df))
print("File exists:", file_path.exists())

CLEAN_DIR = PROJECT_DIR / "data" / "Final_Version" / "Clean"
output_file = CLEAN_DIR / "Facilities_Clean.csv"

print("File exists:", output_file.exists())
dff = pd.read_csv(output_file)
print("Rows saved:", len(dff))

pd.set_option("display.max.rows",None)
pd.set_option("display.max.columns",None)

df.sort_values("LoanID")



File exists: True
Rows saved: 90
File exists: True
File exists: True
Rows saved: 71


,LoanID,FacilityID,DisbursedAmount,InterestRate,LoanStatus
47,3003,2127,NaN,6.5,ACTIVE
79,3003,2068,600000.0,7.2,NaN
28,3009,2188,600000.0,6.5,ACTIVE
35,3012,2098,600000.0,rate,Active
30,3012,2126,600000.0,6.5,NaN
44,3014,2168,250000.0,rate,Active
31,3015,2047,-5000.0,NaN,Closed
36,3016,2199,NaN,7.2,ACTIVE
64,3016,2066,NaN,NaN,ACTIVE
29,3024,2009,NaN,rate,ACTIVE


In [8]:
df["LoanStatus"]=(df["LoanStatus"]
                      .str.strip()
                      .str.title())
df

,LoanID,FacilityID,DisbursedAmount,InterestRate,LoanStatus
0,3069,2184,NaN,7.2,Active
1,3110,2197,NaN,NaN,Closed
2,3040,2191,-5000.0,7.2,Closed
3,3086,2111,250000.0,NaN,NaN
4,3106,2040,250000.0,6.5,Active
5,3125,2167,-5000.0,6.5,Closed
6,3105,2160,-5000.0,6.5,Closed
7,3072,2198,250000.0,7.2,Active
8,3192,2130,-5000.0,NaN,Active
9,3103,2029,250000.0,6.5,Closed


In [9]:
df[df["DisbursedAmount"] < 0].head()

,LoanID,FacilityID,DisbursedAmount,InterestRate,LoanStatus
2,3040,2191,-5000.0,7.2,Closed
5,3125,2167,-5000.0,6.5,Closed
6,3105,2160,-5000.0,6.5,Closed
8,3192,2130,-5000.0,NaN,Active
13,3177,2079,-5000.0,rate,Active


In [10]:
#select * from loans where loanstatus='Active' and DisbursedAmount<0
condition = ((df["LoanStatus"]=="Active") & (df["DisbursedAmount"]<0)) 

#select * from loans where disbursedamount<0 and loanstatus not in ('Active','Closed') and interestrate="rate"
condition2 = ((df["DisbursedAmount"]<0) & (~df["LoanStatus"].isin(["Active","Closed"])) & (df["InterestRate"].isin(["rate"])))

df1=df[condition] #display the df along with condition
df2=df[~condition] #display the df excluding condition
condition3 = (df["InterestRate"]=="rate")

condition4 = df["DisbursedAmount"]<0
df3=df[~(condition|condition2|condition3|condition4)]

df3=df[~(condition|condition2|condition3|condition4)]

df3

df_dirty=df[(condition|condition2|condition3|condition4)]

AUDIT_DIR=PROJECT_DIR/"data"/"Final_Version"/"Audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

audit_file = AUDIT_DIR / "Loans_Audit.csv"

df_dirty.to_csv(audit_file, index=False)
print("Saved successfully:", audit_file)
print("Rows saved:", len(df_dirty))

#no_of_bad_rows=(condition|condition2|condition3|condition4).sum()

#print("Rows to be removed:", no_of_bad_rows)
df=df3
df
print("Rows saved:", len(df))


Saved successfully: C:\Users\Madhura J\Desktop\Data Analyst Banking Project\data\Final_Version\Audit\Loans_Audit.csv
Rows saved: 40
Rows saved: 50


In [11]:
df["FacilityID"].isin(dff["FacilityID"]) 
df["LoanID"].is_unique

False

In [12]:
duplicates = df[df.duplicated("LoanID", keep=False)] \
            .sort_values("LoanID")
duplicates
df["DuplicatedLoanIDFlag"]=df.duplicated(subset="LoanID", keep=False)

C:\Users\Madhura J\AppData\Local\Temp\ipykernel_27960\914832715.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["DuplicatedLoanIDFlag"]=df.duplicated(subset="LoanID", keep=False)


In [13]:
df=df.copy()
df["DuplicatedLoanIDFlag"] = df.duplicated(subset="LoanID", keep=False)
df


,LoanID,FacilityID,DisbursedAmount,InterestRate,LoanStatus,DuplicatedLoanIDFlag
0,3069,2184,NaN,7.2,Active,False
1,3110,2197,NaN,NaN,Closed,False
3,3086,2111,250000.0,NaN,NaN,False
4,3106,2040,250000.0,6.5,Active,False
7,3072,2198,250000.0,7.2,Active,False
9,3103,2029,250000.0,6.5,Closed,False
10,3161,2101,NaN,7.2,Active,False
11,3147,2090,600000.0,6.5,Active,False
12,3112,2168,NaN,NaN,Closed,False
17,3101,2151,NaN,NaN,Closed,False


In [14]:
df["FacilitiesExistsFlag"] = df["FacilityID"].isin(dff["FacilityID"])
df

,LoanID,FacilityID,DisbursedAmount,InterestRate,LoanStatus,DuplicatedLoanIDFlag,FacilitiesExistsFlag
0,3069,2184,NaN,7.2,Active,False,True
1,3110,2197,NaN,NaN,Closed,False,False
3,3086,2111,250000.0,NaN,NaN,False,True
4,3106,2040,250000.0,6.5,Active,False,False
7,3072,2198,250000.0,7.2,Active,False,True
9,3103,2029,250000.0,6.5,Closed,False,True
10,3161,2101,NaN,7.2,Active,False,False
11,3147,2090,600000.0,6.5,Active,False,False
12,3112,2168,NaN,NaN,Closed,False,True
17,3101,2151,NaN,NaN,Closed,False,False


In [15]:
PROJECT_DIR = Path.cwd().parent

CLEAN_DIR = PROJECT_DIR / "data" / "Final_Version" / "Clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

output_file = CLEAN_DIR / "Loans_Clean.csv"

df.to_csv(output_file, index=False)

print("Saved successfully:", output_file)
print("Rows saved:", len(df))

Saved successfully: C:\Users\Madhura J\Desktop\Data Analyst Banking Project\data\Final_Version\Clean\Loans_Clean.csv
Rows saved: 50
